# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [ ]:
# !pip install tensorboardX
# I hate jupyter files. They have taken ML back by a decade. 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
# train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
#     [transforms.ToTensor()]
# ))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [3]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=use_cuda, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

In [4]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    original = x.detach()
    adv_x = original.clone()

    for _ in range(k):
        adv_x.requires_grad_()
        loss = F.cross_entropy(model(adv_x), labels)
        grad, = torch.autograd.grad(loss, adv_x)

        # Increase the loss for the true class.
        adv_x = adv_x.detach() + eps_step * grad.sign()
        delta = (adv_x - original).clamp(-eps, eps)
        adv_x = (original + delta).clamp(0, 1)

    return adv_x.detach()


In [5]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    original = x.detach()
    adv_x = original.clone()

    for _ in range(k):
        adv_x.requires_grad_()
        loss = F.cross_entropy(model(adv_x), labels)
        grad, = torch.autograd.grad(loss, adv_x)

        # Calculate a separate gradient length for each image.
        grad_norm = grad.flatten(1).norm(p=2, dim=1).view(-1, 1, 1, 1)
        adv_x = adv_x.detach() + eps_step * grad / (grad_norm + 1e-10)

        # Shrink perturbations whose L2 length exceeds eps.
        delta = adv_x - original
        delta_norm = delta.flatten(1).norm(p=2, dim=1).view(-1, 1, 1, 1)
        delta = delta * (eps / delta_norm.clamp(min=1e-10)).clamp(max=1)
        adv_x = (original + delta).clamp(0, 1)

    return adv_x.detach()


# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [6]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1, k=20):
    model.eval()
    correct = 0
    total = 0
    for x, labels in tqdm(test_loader):
        x, labels = x.to(device), labels.to(device)
        if attack == 'pgd_linf':
            adv_x = pgd_linf_untargeted(model, x, labels, k, eps, eps / 4)
        elif attack == 'pgd_l2':
            adv_x = pgd_l2_untargeted(model, x, labels, k, eps, eps / 4)
        else:
            raise ValueError("Unknown attack: " + attack)

        with torch.no_grad():
            clean_correct = model(x).argmax(dim=1) == labels
            adv_correct = model(adv_x).argmax(dim=1) == labels
        correct += (clean_correct & adv_correct).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    print(f"{attack}: {accuracy:.2%} (eps={eps}, steps={k})")
    return accuracy


def test_clean_accuracy(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, labels in test_loader:
            x, labels = x.to(device), labels.to(device)
            correct += (model(x).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"Clean accuracy: {accuracy:.2%}")
    return accuracy


## Single-Norm Robust Accuracy

In [7]:
# Put the three supplied checkpoints in models/ next to this notebook.
# All attacks use 20 steps and a step size of eps / 4.
results = {}
for name in ('Linf', 'L2', 'RAMP'):
    print("\nModel:", name)
    model.load_state_dict(torch.load(f'models/pretr_{name}.pth',
                                    map_location=device, weights_only=True))
    results[name] = {}
    results[name]['clean'] = test_clean_accuracy(model)
    results[name]['linf'] = test_model_on_single_attack(
        model, attack='pgd_linf', eps=8/255)



Model: Linf
Clean accuracy: 82.80%


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:20<00:00,  7.60it/s]


pgd_linf: 50.76% (eps=0.03137254901960784, steps=20)

Model: L2
Clean accuracy: 88.75%


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:22<00:00,  6.97it/s]


pgd_linf: 29.47% (eps=0.03137254901960784, steps=20)

Model: RAMP
Clean accuracy: 81.19%


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:21<00:00,  7.25it/s]

pgd_linf: 49.06% (eps=0.03137254901960784, steps=20)


In [8]:
for name in ('Linf', 'L2', 'RAMP'):
    print("\nModel:", name)
    model.load_state_dict(torch.load(f'models/pretr_{name}.pth',
                                    map_location=device, weights_only=True))
    results[name]['l2'] = test_model_on_single_attack(
        model, attack='pgd_l2', eps=0.75)



Model: Linf


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:21<00:00,  7.16it/s]


pgd_l2: 47.02% (eps=0.75, steps=20)

Model: L2


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:22<00:00,  6.86it/s]


pgd_l2: 53.58% (eps=0.75, steps=20)

Model: RAMP


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:21<00:00,  7.26it/s]

pgd_l2: 59.37% (eps=0.75, steps=20)


## Multi-Norm Robust Accuracy

In [9]:
def test_model_on_multi_attacks(model, eps_linf=8/255, eps_l2=0.75, k=20):
    model.eval()
    correct = 0
    total = 0
    for x, labels in tqdm(test_loader):
        x, labels = x.to(device), labels.to(device)
        adv_linf = pgd_linf_untargeted(model, x, labels, k, eps_linf, eps_linf / 4)
        adv_l2 = pgd_l2_untargeted(model, x, labels, k, eps_l2, eps_l2 / 4)

        with torch.no_grad():
            clean_correct = model(x).argmax(dim=1) == labels
            linf_correct = model(adv_linf).argmax(dim=1) == labels
            l2_correct = model(adv_l2).argmax(dim=1) == labels
        # The SAME image must survive both attacks.
        correct += (clean_correct & linf_correct & l2_correct).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    print(f"Union accuracy: {accuracy:.2%}")
    return accuracy


In [10]:
for name in ('Linf', 'L2', 'RAMP'):
    print("\nModel:", name)
    model.load_state_dict(torch.load(f'models/pretr_{name}.pth',
                                    map_location=device, weights_only=True))
    results[name]['union'] = test_model_on_multi_attacks(model)

print("\nModel    Clean     Linf      L2        Union")
for name, scores in results.items():
    print(f"{name:8} {scores['clean']:.2%}    {scores['linf']:.2%}    "
          f"{scores['l2']:.2%}    {scores['union']:.2%}")



Model: Linf


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:41<00:00,  3.81it/s]


Union accuracy: 45.53%

Model: L2


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:42<00:00,  3.71it/s]


Union accuracy: 29.47%

Model: RAMP


100%|██████████████████████████████████████████████████████████████████| 157/157 [00:40<00:00,  3.89it/s]

Union accuracy: 49.05%

Model    Clean     Linf      L2        Union
Linf     82.80%    50.76%    47.02%    45.53%
L2       88.75%    29.47%    53.58%    29.47%
RAMP     81.19%    49.06%    59.37%    49.05%
